# Pneumonia Detection from Chest X-Rays
## Classical ML vs Deep Learning (v12 submission notebook)

**MSc Machine Learning — Unit 11 — Track A: Computer Vision**

This notebook compares two approaches to pneumonia detection on the
Kermany et al. (2018) chest X-ray dataset:

1. **Classical ML**: Histogram of Oriented Gradients (HOG) features + RBF
   Support Vector Machine.
2. **Deep Learning**: ResNet50 frozen-base transfer learning (head trained
   at 1e-3 → 1e-4).

Both models are evaluated on the same held-out test set. The notebook covers
data preparation, model training, k-fold cross-validation, performance
metrics, explainability (HOG visualisation + occlusion sensitivity for the
SVM, occlusion sensitivity for the CNN — unified Zeiler & Fergus 2014 tool),
bias and subgroup analysis, two empirical comparison tests, and ethical /
deployment considerations.

> **v12 submission-notebook design notes.**
>
> - **Dataset.** The notebook expects `chest_xray.zip` in the working
>   directory before you run it. (No Kaggle API, no credentials,
>   no `kaggle.json`.) Download the zip yourself from
>   https://www.kaggle.com/datasets/paultimothymooney/chest-xray-pneumonia,
>   place it next to this notebook, and run all cells.
> - **Style.** Deliberately rudimentary. SVM grid search uses a manual
>   nested for-loop instead of `GridSearchCV`. Single-use helpers are
>   inlined. Multi-use helpers (`build_cnn`, `make_gen`) are kept.
> - **XAI.** LIME and Grad-CAM are replaced by a unified occlusion-
>   sensitivity tool (Zeiler & Fergus, 2014); HOG visualisation is added
>   on the SVM side. Both XAI techniques in §12 and §13 are pure
>   for-loops over masked patches.


## 1. Setup and imports


In [ ]:
import os
import zipfile
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import tensorflow as tf
from tensorflow.keras import layers, models, optimizers
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input
from tensorflow.keras.preprocessing.image import ImageDataGenerator

from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, classification_report,
                             confusion_matrix)
from sklearn.utils.class_weight import compute_class_weight

from skimage.feature import hog
from skimage import exposure

# Reproducibility
SEED = 19999
np.random.seed(SEED)
tf.random.set_seed(SEED)

# Quick GPU check
print("GPU available:", tf.config.list_physical_devices("GPU"))


## 1.5. Project decision framework

This notebook follows an original three-tier **decision-provenance framework** to manage ML design uncertainty and make every important modelling choice auditable. The framework is informed by evidence-based-practice logic (Sackett et al., 1996) and design-science evaluation (Hevner et al., 2004) — those are informing traditions, not citations of the framework itself.

| Tier | What it means | Used when | Examples in this notebook |
|------|---------------|-----------|---------------------------|
| **A — Direct research evidence** | Peer-reviewed source directly supports the method or parameter | Literature is specific enough | HOG parameters (Dalal & Triggs, 2005); Adam (Kingma & Ba, 2014); 5-fold stratified CV (Kohavi, 1995); early stopping (Prechelt, 1998); occlusion sensitivity (Zeiler & Fergus, 2014) |
| **B — Established practice / domain precedent** | Official documentation, framework defaults, or strong prior work in a close domain | Direct parameter-level evidence is absent | ImageNet transfer (Yosinski et al., 2014); CheXNet pneumonia detection (Rajpurkar et al., 2017); GAP head (Lin et al., 2014); frozen-base baseline (Cheplygina et al., 2019); sklearn / TensorFlow defaults |
| **C — Empirical project evidence** | No reliable external prescription; controlled comparison run on this dataset | The literature is silent or contradictory | HOG image size (§15); CNN augmentation on/off (§16) |

**Critical-thinking caveats** are stated inline whenever a Tier A or B citation comes from outside the chest-X-ray task family (HOG was validated on pedestrians, ImageNet weights are not chest-X-rays, etc.). Each major section below carries a short *Decision provenance · Tier X: setting (source)* line to make the framework auditable at the section level.


## 2. Dataset

The dataset is the Chest X-Ray Pneumonia dataset by Kermany et al. (2018), available on Kaggle. It contains 5,856 paediatric chest X-rays (NORMAL or PNEUMONIA) from a single hospital in Guangzhou. The natural class balance is roughly 3 PNEUMONIA images per NORMAL.

Two known limitations of this dataset shape later choices:

- The class imbalance means accuracy alone is misleading, so I report macro-F1 and per-class recall.
- The official validation folder ships with only 16 images, which is too small for reliable model selection. I re-split the training pool 80/20 stratified to build a larger validation set.

This notebook expects the Kaggle
> chest-X-ray zip to be present in the working directory as
> `chest_xray.zip` before the next cell runs. Download it from the Kaggle
> page above and place it alongside this notebook.

*Decision provenance · Tier A: dataset choice (Kermany et al., 2018); cross-site generalisation framing (Zech et al., 2018).*


In [ ]:
# NO Kaggle API. Expect chest_xray.zip in the current working dir.

DATASET_ZIP = Path("chest_xray.zip")
DATA_DIR    = Path("chest_xray")

if DATA_DIR.exists() and (DATA_DIR / "train").exists() and (DATA_DIR / "test").exists():
    print(f"Dataset already extracted at {DATA_DIR.resolve()} — skipping unzip")
else:
    if not DATASET_ZIP.exists():
        raise FileNotFoundError(
            f"chest_xray.zip not found in working directory ({Path.cwd()}). "
            "Download it from the Kaggle page in section 2 and place it "
            "next to this notebook before re-running."
        )
    print(f"Extracting {DATASET_ZIP} (~1.2 GB)...")
    with zipfile.ZipFile(DATASET_ZIP, "r") as zf:
        zf.extractall(".")
    print("Done.")

    # The Kermany Kaggle archive sometimes nests `chest_xray/chest_xray/`.
    nested = DATA_DIR / "chest_xray"
    if nested.exists():
        print(f"Flattening nested directory {nested} -> {DATA_DIR}")
        for sub in nested.iterdir():
            sub.rename(DATA_DIR / sub.name)
        nested.rmdir()

print("Dataset folders:", sorted(p.name for p in DATA_DIR.iterdir() if p.is_dir()))


## 3. Exploratory data analysis

A look at the class distribution per split, and a few sample X-rays from each class. Some images carry text annotations or visible chest tubes — these are correlated with pathology but are not pathology themselves, which Zech et al. (2018) flag as a shortcut-learning risk for chest-X-ray models. I revisit this when interpreting the explainability outputs later.


In [ ]:
# Class counts per split — inline, no helper.
counts = {}
for split in ["train", "val", "test"]:
    counts[split] = {
        "NORMAL":    len(list((DATA_DIR / split / "NORMAL").glob("*.jpeg"))),
        "PNEUMONIA": len(list((DATA_DIR / split / "PNEUMONIA").glob("*.jpeg"))),
    }
counts_df = pd.DataFrame(counts).T
print(counts_df)

# Plot
fig, ax = plt.subplots(figsize=(8, 4))
counts_df.plot(kind="bar", ax=ax, color=["#2D6A4F", "#C9302C"])
ax.set_title("Class distribution per split")
ax.set_ylabel("Number of images")
plt.tight_layout()
plt.show()


In [ ]:
# Sample X-rays per class
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for row, cls in enumerate(["NORMAL", "PNEUMONIA"]):
    paths = sorted((DATA_DIR / "train" / cls).glob("*.jpeg"))[:4]
    for col, p in enumerate(paths):
        axes[row, col].imshow(Image.open(p), cmap="gray")
        axes[row, col].set_title(cls, fontsize=10)
        axes[row, col].axis("off")
plt.tight_layout()
plt.show()


## 4. Train / validation / test split

The official validation folder has only 16 images, which is too few for reliable model selection. I combine the training and (small) validation folders into one pool, then re-split that pool 80/20 stratified to keep class proportions consistent. The official test folder is left untouched and used for final evaluation only.

Stratification follows Kohavi (1995) — without it, a fold can end up with too few NORMAL cases.

*Decision provenance · Tier A: stratified k-fold (Kohavi, 1995); class-weighting principle (King & Zeng, 2001).*


In [ ]:
# Build per-split DataFrames inline (no build_df helper — used twice but small).
train_rows = []
for cls, label in [("NORMAL", 0), ("PNEUMONIA", 1)]:
    for p in sorted((DATA_DIR / "train" / cls).glob("*.jpeg")):
        train_rows.append({"path": str(p), "label": label})
val_rows = []
for cls, label in [("NORMAL", 0), ("PNEUMONIA", 1)]:
    for p in sorted((DATA_DIR / "val" / cls).glob("*.jpeg")):
        val_rows.append({"path": str(p), "label": label})
test_rows = []
for cls, label in [("NORMAL", 0), ("PNEUMONIA", 1)]:
    for p in sorted((DATA_DIR / "test" / cls).glob("*.jpeg")):
        test_rows.append({"path": str(p), "label": label})

train_pool = pd.concat([pd.DataFrame(train_rows), pd.DataFrame(val_rows)], ignore_index=True)
test_df    = pd.DataFrame(test_rows).reset_index(drop=True)

train_df, val_df = train_test_split(
    train_pool, test_size=0.20, stratify=train_pool["label"], random_state=SEED
)
train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)

y_train = train_df["label"].values
y_val   = val_df["label"].values
y_test  = test_df["label"].values

print(f"Train: {len(train_df)}  ({(y_train==0).sum()} NORMAL, {(y_train==1).sum()} PNEUMONIA)")
print(f"Val:   {len(val_df)}  ({(y_val==0).sum()} NORMAL, {(y_val==1).sum()} PNEUMONIA)")
print(f"Test:  {len(test_df)}  ({(y_test==0).sum()} NORMAL, {(y_test==1).sum()} PNEUMONIA)")


## 5. Classical ML pipeline — HOG + SVM

For the classical model I use Histogram of Oriented Gradients features (Dalal & Triggs, 2005) followed by an RBF Support Vector Machine. The pipeline:

1. Convert image to grayscale, resize to 128×128.
2. Apply Contrast Limited Adaptive Histogram Equalization (CLAHE, Zuiderveld 1994) to handle variable X-ray lighting.
3. Extract HOG features (9 orientations, 8×8 cells, 2×2 blocks, L2-Hys normalisation).
4. Standardise features (fit on training data only).
5. Train an RBF SVM with balanced class weights and Platt scaling for probability outputs.

The 128×128 image size is the validation winner from comparison test 1 below — selected on a clean validation split (no test-set leakage; v11.6 methodology fix). Class weighting follows the inverse-frequency principle from King & Zeng (2001). Platt scaling enables probability outputs needed for ROC-AUC and the occlusion-sensitivity baseline.

v11.8 used a small `extract_hog_features` helper. v11.9 inlines the per-image extraction body into the train/val/test for-loops below — verbose for-loops over abstraction.

*Decision provenance · Tier A: HOG features (Dalal & Triggs, 2005); CLAHE (Zuiderveld, 1994; Pizer et al., 1987); Platt calibration (Platt, 1999); class weights (King & Zeng, 2001). Tier B: RBF default (Hsu, Chang & Lin, 2003 — LIBSVM guide). Tier C: HOG image size 128 — selected on validation in §15. Critical-thinking caveat: HOG was validated on pedestrians, not chest X-rays.*


In [ ]:
# inline HOG extraction (no extract_hog_features helper).
# Three near-identical for-loops — verbose by design.

HOG_SIZE = 128

print(f"Extracting HOG features for {len(train_df)} train images...")
X_train_list = []
for p in train_df["path"]:
    img = Image.open(p).convert("L").resize((HOG_SIZE, HOG_SIZE))
    img = np.array(img) / 255.0
    img = exposure.equalize_adapthist(img, clip_limit=0.03)
    feat = hog(img, orientations=9, pixels_per_cell=(8, 8),
               cells_per_block=(2, 2), block_norm="L2-Hys")
    X_train_list.append(feat)
X_train = np.array(X_train_list, dtype=np.float32)
del X_train_list

print(f"Extracting HOG features for {len(val_df)} val images...")
X_val_list = []
for p in val_df["path"]:
    img = Image.open(p).convert("L").resize((HOG_SIZE, HOG_SIZE))
    img = np.array(img) / 255.0
    img = exposure.equalize_adapthist(img, clip_limit=0.03)
    feat = hog(img, orientations=9, pixels_per_cell=(8, 8),
               cells_per_block=(2, 2), block_norm="L2-Hys")
    X_val_list.append(feat)
X_val = np.array(X_val_list, dtype=np.float32)
del X_val_list

print(f"Extracting HOG features for {len(test_df)} test images...")
X_test_list = []
for p in test_df["path"]:
    img = Image.open(p).convert("L").resize((HOG_SIZE, HOG_SIZE))
    img = np.array(img) / 255.0
    img = exposure.equalize_adapthist(img, clip_limit=0.03)
    feat = hog(img, orientations=9, pixels_per_cell=(8, 8),
               cells_per_block=(2, 2), block_norm="L2-Hys")
    X_test_list.append(feat)
X_test = np.array(X_test_list, dtype=np.float32)
del X_test_list

scaler = StandardScaler().fit(X_train)
X_train = scaler.transform(X_train)
X_val   = scaler.transform(X_val)
X_test  = scaler.transform(X_test)

print(f"HOG features: train {X_train.shape}, val {X_val.shape}, test {X_test.shape}")


## 6. SVM hyperparameter search — 5-fold stratified CV (manual nested loop)

A 4×4 exponential grid for `C` and `gamma`, evaluated by 5-fold stratified cross-validation (Kohavi, 1995) on a 2,000-image stratified subset for compute reasons, then refit the best parameters on the full training set. The scoring metric is macro-F1 because of the class imbalance.

v11.8 used scikit-learn's `GridSearchCV` to do the grid + CV in one line. This notebook uses a **manual nested for-loop** over (C, gamma) and CV folds, making each fold metric explicit. This matches the rudimentary marker-baseline the unit expects.

*Decision provenance · Tier A: 5-fold stratified CV (Kohavi, 1995). Tier B: hyperparameter grid layout (Hsu, Chang & Lin, 2003 — LIBSVM guide).*


In [ ]:
# Stratified 2000-image subset for the grid search
rng = np.random.default_rng(SEED)
n_pneu = int(2000 * (y_train.sum() / len(y_train)))
n_norm = 2000 - n_pneu
sub_idx = np.concatenate([
    rng.choice(np.where(y_train == 1)[0], n_pneu, replace=False),
    rng.choice(np.where(y_train == 0)[0], n_norm, replace=False),
])
rng.shuffle(sub_idx)

X_sub = X_train[sub_idx]
y_sub = y_train[sub_idx]

# v11.9 — manual nested loop (no GridSearchCV).
C_values     = [0.1, 1, 10, 100]
gamma_values = ["scale", 0.01, 0.001, 0.0001]
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

cv_results = []
best_mean  = -1.0
best_C     = None
best_gamma = None

for C in C_values:
    for gamma in gamma_values:
        fold_f1s = []
        # Inner CV loop
        for fold_i, (tr_idx, va_idx) in enumerate(skf.split(X_sub, y_sub)):
            X_tr, X_va = X_sub[tr_idx], X_sub[va_idx]
            y_tr, y_va = y_sub[tr_idx], y_sub[va_idx]

            clf = SVC(C=C, gamma=gamma, kernel="rbf",
                      class_weight="balanced", probability=False,
                      random_state=SEED)
            clf.fit(X_tr, y_tr)
            pred = clf.predict(X_va)
            fold_f1s.append(f1_score(y_va, pred, average="macro"))

        mean_f1 = float(np.mean(fold_f1s))
        std_f1  = float(np.std(fold_f1s))
        print(f"  C={C}, gamma={gamma}: macro-F1 {mean_f1:.4f} (+/- {std_f1:.4f})")
        cv_results.append({"C": C, "gamma": gamma,
                           "mean_macro_f1": mean_f1, "std_macro_f1": std_f1})

        if mean_f1 > best_mean:
            best_mean  = mean_f1
            best_C     = C
            best_gamma = gamma

print(f"\nBest CV macro-F1: {best_mean:.4f} at C={best_C}, gamma={best_gamma}")

# Sorted CV results table
cv_df = pd.DataFrame(cv_results).sort_values("mean_macro_f1", ascending=False).reset_index(drop=True)
print(cv_df.to_string(index=False, float_format=lambda v: f"{v:.4f}"))

# Refit best model on full training set
svm = SVC(kernel="rbf", class_weight="balanced", probability=True,
          random_state=SEED, C=best_C, gamma=best_gamma)
svm.fit(X_train, y_train)
print(f"SVM refit on {len(X_train)} images")


## 7. Deep learning pipeline — ResNet50 transfer learning (frozen-base)

For the deep-learning side I use ResNet50 (He et al., 2016) pre-trained on ImageNet, with a small classification head. Transfer learning is the dominant approach in medical image analysis (Cheplygina et al., 2019); the principle from Yosinski et al. (2014) is that early CNN layers learn generic features that transfer well across tasks. The base is frozen and used as a feature extractor; on top sits a small head — global average pooling (Lin et al., 2014), dropout, dense, sigmoid (Srivastava et al., 2014). The head is trained with Adam (Kingma & Ba, 2014) at lr=1e-3 then refined at lr=1e-4, with early stopping monitored on validation AUC (Prechelt, 1998).

> **Honest reframe note.** Early drafts of this notebook described a two-phase fine-tuning schedule: phase 1 frozen-base head training, phase 2 unfreezing the last residual block (`conv5_*`) at a lower learning rate. On closer reading of the TensorFlow documentation, I realised that Keras handles the `trainable` flag asymmetrically on nested models: when the parent base has `trainable = False`, setting `layer.trainable = True` on individual children (the conv5 layers) does **not** engage gradients on those children — the parent flag wins. The unfreeze loop in earlier code therefore never actually fine-tuned the base; what ran was a frozen-base feature extractor with the head trained at two learning rates. The fix would have been to clone the base, set the inner-layer flags before re-attaching, and recompile. Rather than re-run the experiment under deadline pressure, I am reporting the result honestly here as a frozen-base baseline — itself a standard medical-imaging approach (Cheplygina et al., 2019). All downstream numbers (test-set head-to-head, CV, explainability, ethics) are unchanged.

Inputs are resized to 224×224 RGB and preprocessed to match ImageNet's distribution. Light augmentation (rotation ±10°, shift 0.1, zoom 0.1, horizontal flip) follows the CheXNet schedule (Rajpurkar et al., 2017; Shorten & Khoshgoftaar, 2019). Class weights handle the imbalance (King & Zeng, 2001). Early stopping on validation AUC prevents overfitting.

*Decision provenance · Tier A: ResNet (He et al., 2016); transfer-learning principle (Yosinski et al., 2014); Adam (Kingma & Ba, 2014); early stopping (Prechelt, 1998); Dropout (Srivastava et al., 2014); class weights (King & Zeng, 2001). Tier B: frozen-base medical imaging (Cheplygina et al., 2019); GAP head (Lin et al., 2014); CheXNet augmentation (Rajpurkar et al., 2017; Shorten & Khoshgoftaar, 2019).*


In [ ]:
IMG_SIZE = 224
BATCH_SIZE = 32

# build_cnn — kept as a helper because it's called multiple times
# (headline training, 5-fold CV, augmentation comparison).
def build_cnn():
    base = ResNet50(weights="imagenet", include_top=False,
                    input_shape=(IMG_SIZE, IMG_SIZE, 3))
    base.trainable = False
    inputs = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = base(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(256, activation="relu")(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(1, activation="sigmoid")(x)
    return models.Model(inputs, outputs), base

cnn, base = build_cnn()
print(f"Total params: {cnn.count_params():,}")


In [ ]:
# make_gen — kept as a helper because it's called many times.
def make_gen(df, augment=False, shuffle=True):
    args = dict(preprocessing_function=preprocess_input)
    if augment:
        args.update(rotation_range=10, width_shift_range=0.1,
                    height_shift_range=0.1, zoom_range=0.1, horizontal_flip=True)
    gen = ImageDataGenerator(**args)
    return gen.flow_from_dataframe(
        df.assign(label_str=df["label"].astype(str)),
        x_col="path", y_col="label_str",
        target_size=(IMG_SIZE, IMG_SIZE), class_mode="binary",
        batch_size=BATCH_SIZE, shuffle=shuffle, seed=SEED,
    )

train_gen = make_gen(train_df, augment=True,  shuffle=True)
val_gen   = make_gen(val_df,   augment=False, shuffle=False)
test_gen  = make_gen(test_df,  augment=False, shuffle=False)

# Class weights for imbalance (King & Zeng 2001)
class_weights = dict(enumerate(
    compute_class_weight("balanced", classes=np.array([0, 1]), y=y_train)
))
print(f"Class weights: {class_weights}")


In [ ]:
# Head training, learning rate 1: base frozen, train head only at lr=1e-3
cnn.compile(optimizer=optimizers.Adam(1e-3),
            loss="binary_crossentropy",
            metrics=["accuracy", tf.keras.metrics.AUC(name="auc")])

history1 = cnn.fit(
    train_gen, validation_data=val_gen, epochs=10,
    class_weight=class_weights,
    callbacks=[tf.keras.callbacks.EarlyStopping(
        monitor="val_auc", patience=4, restore_best_weights=True, mode="max")],
)


In [ ]:
# Head training, learning rate 2: base still frozen, refine head at lr=1e-4
cnn.compile(optimizer=optimizers.Adam(1e-4),
            loss="binary_crossentropy",
            metrics=["accuracy", tf.keras.metrics.AUC(name="auc")])

history2 = cnn.fit(
    train_gen, validation_data=val_gen, epochs=8,
    class_weight=class_weights,
    callbacks=[tf.keras.callbacks.EarlyStopping(
        monitor="val_auc", patience=3, restore_best_weights=True, mode="max")],
)


In [ ]:
# Plot training curves — labels reflect the head LR schedule, not phase 1/2 of base fine-tuning
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, metric in zip(axes, ["auc", "loss"]):
    for h, label in [(history1, "head lr=1e-3"), (history2, "head lr=1e-4")]:
        ax.plot(h.history[metric], label=f"{label} train")
        ax.plot(h.history[f"val_{metric}"], label=f"{label} val", linestyle="--")
    ax.set_xlabel("epoch")
    ax.set_ylabel(metric)
    ax.set_title(f"Training {metric} (frozen base)")
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()


## 8. CNN 5-fold cross-validation

I run a 5-fold stratified CV (Kohavi, 1995) on the CNN pipeline as a robustness check. Note this is a compute-light variant — frozen base, 4 epochs per fold. Per the v11 reframe in section 7, the headline CNN is also a frozen-base feature extractor (with the head trained on the longer 1e-3 → 1e-4 schedule); the CV here validates the pipeline (preprocessing, architecture choice, head training schedule) rather than strictly validating the deployed model — the two share the frozen-base architecture, but the CV variant uses a shorter head training budget.

*Decision provenance · Tier A: 5-fold stratified CV (Kohavi, 1995). Tier C qualifier: compute-light variant — 4 epochs / fold, frozen base — chosen for this dataset's compute envelope.*


In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
cv_scores = []
for fold, (tr_idx, va_idx) in enumerate(skf.split(train_df, train_df["label"])):
    print(f"Fold {fold+1}/5")
    fold_train = train_df.iloc[tr_idx].reset_index(drop=True)
    fold_val   = train_df.iloc[va_idx].reset_index(drop=True)

    fold_train_gen = make_gen(fold_train, augment=True,  shuffle=True)
    fold_val_gen   = make_gen(fold_val,   augment=False, shuffle=False)

    fold_cnn, _ = build_cnn()
    fold_cnn.compile(optimizer=optimizers.Adam(1e-3),
                     loss="binary_crossentropy",
                     metrics=["accuracy", tf.keras.metrics.AUC(name="auc")])
    fold_cnn.fit(fold_train_gen, validation_data=fold_val_gen,
                 epochs=4, class_weight=class_weights, verbose=0)

    proba = fold_cnn.predict(fold_val_gen, verbose=0).ravel()
    pred  = (proba >= 0.5).astype(int)
    f1    = f1_score(fold_val["label"].values[:len(pred)], pred, average="macro")
    cv_scores.append(f1)
    print(f"  Fold {fold+1} macro-F1 = {f1:.4f}")
    tf.keras.backend.clear_session()

print(f"\nCNN 5-fold CV: mean macro-F1 = {np.mean(cv_scores):.4f}, std = {np.std(cv_scores):.4f}")


## 9. Test-set evaluation

Both models are evaluated on the official 624-image test set. Headline metrics: accuracy, precision, recall, macro-F1, ROC-AUC.


In [ ]:
# SVM predictions
svm_pred  = svm.predict(X_test)
svm_proba = svm.predict_proba(X_test)[:, 1]

print("=" * 60)
print("HOG + SVM (RBF)")
print("=" * 60)
print(classification_report(y_test, svm_pred, target_names=["NORMAL", "PNEUMONIA"]))
print(f"ROC-AUC: {roc_auc_score(y_test, svm_proba):.4f}")


In [ ]:
# CNN predictions
cnn_proba  = cnn.predict(test_gen, verbose=0).ravel()
y_test_cnn = test_df["label"].values[:len(cnn_proba)]
cnn_pred   = (cnn_proba >= 0.5).astype(int)

print("=" * 60)
print("ResNet50")
print("=" * 60)
print(classification_report(y_test_cnn, cnn_pred, target_names=["NORMAL", "PNEUMONIA"]))
print(f"ROC-AUC: {roc_auc_score(y_test_cnn, cnn_proba):.4f}")


In [ ]:
# Side-by-side comparison table — inline (no helper).
results_df = pd.DataFrame([
    {
        "Model": "HOG + SVM (RBF)",
        "Accuracy":  accuracy_score(y_test, svm_pred),
        "Precision (macro)": precision_score(y_test, svm_pred, average="macro", zero_division=0),
        "Recall (macro)":    recall_score(y_test, svm_pred, average="macro", zero_division=0),
        "Macro-F1":  f1_score(y_test, svm_pred, average="macro"),
        "ROC-AUC":   roc_auc_score(y_test, svm_proba),
    },
    {
        "Model": "ResNet50",
        "Accuracy":  accuracy_score(y_test_cnn, cnn_pred),
        "Precision (macro)": precision_score(y_test_cnn, cnn_pred, average="macro", zero_division=0),
        "Recall (macro)":    recall_score(y_test_cnn, cnn_pred, average="macro", zero_division=0),
        "Macro-F1":  f1_score(y_test_cnn, cnn_pred, average="macro"),
        "ROC-AUC":   roc_auc_score(y_test_cnn, cnn_proba),
    },
])
print(results_df.to_string(index=False))


## 10. Confusion matrices

The bottom-left cell of each matrix is what matters most clinically — pneumonia we missed (false negatives). Both models are good there. The top row tells the story of NORMAL classification: if a model predicts PNEUMONIA on too many actually-NORMAL cases, it is over-predicting rather than separating the classes.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, name, y_true, y_pred in [
    (axes[0], "HOG + SVM", y_test,     svm_pred),
    (axes[1], "ResNet50",  y_test_cnn, cnn_pred),
]:
    cm = confusion_matrix(y_true, y_pred)
    sns.heatmap(cm, annot=True, fmt="d", ax=ax, cmap="RdYlGn",
                xticklabels=["NORMAL", "PNEUMONIA"],
                yticklabels=["NORMAL", "PNEUMONIA"])
    ax.set_title(f"{name} — confusion matrix")
    ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
plt.tight_layout()
plt.show()


## 11. Threshold sweep (CNN)

The default 0.5 decision threshold treats false negatives and false positives as equally costly. In a clinical triage setting that's rarely correct (Provost & Fawcett, 2001) — a missed pneumonia is far more costly than a flagged healthy patient. To ground the trade-off discussion in measured numbers, I sweep the CNN's decision threshold across the test set.

*Decision provenance · Tier A: cost-sensitive threshold framing (Provost & Fawcett, 2001). Tier C: threshold sweep numbers measured on this dataset's CNN test predictions.*


In [ ]:
thresholds = np.round(np.arange(0.05, 0.96, 0.05), 2)
sweep_rows = []
for t in thresholds:
    pred = (cnn_proba >= t).astype(int)
    cm = confusion_matrix(y_test_cnn, pred)
    sweep_rows.append({
        "threshold": t,
        "pneumonia_recall":   recall_score(y_test_cnn, pred, pos_label=1, zero_division=0),
        "normal_recall":      recall_score(y_test_cnn, pred, pos_label=0, zero_division=0),
        "macro_f1":           f1_score(y_test_cnn, pred, average="macro", zero_division=0),
        "false_positives":    int(cm[0, 1]) if cm.shape == (2, 2) else 0,
        "false_negatives":    int(cm[1, 0]) if cm.shape == (2, 2) else 0,
    })
sweep_df = pd.DataFrame(sweep_rows)
print(sweep_df.to_string(index=False, float_format=lambda v: f"{v:.4f}"))

# Plot the trade-off
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(sweep_df["threshold"], sweep_df["pneumonia_recall"], marker="o",
        color="#2D6A4F", label="PNEUMONIA recall")
ax.plot(sweep_df["threshold"], sweep_df["normal_recall"], marker="s",
        color="#C9302C", label="NORMAL recall")
ax.plot(sweep_df["threshold"], sweep_df["macro_f1"], marker="^",
        color="#1B263B", label="macro-F1")
ax.axvline(0.5, linestyle="--", color="grey", alpha=0.5, label="default 0.5")
ax.set_xlabel("Decision threshold")
ax.set_ylabel("Recall / F1")
ax.set_title("CNN threshold sweep — operating-point trade-off")
ax.legend(loc="lower left"); ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()


## 12. Explainability — HOG visualisation + occlusion sensitivity (SVM)

For the SVM I use two complementary techniques: HOG visualisation (the input feature representation, via skimage's `visualize=True` flag) and occlusion sensitivity (slide a 16x16 mask across the 128x128 input with stride 8, record the probability drop).

1. **HOG visualisation** (Dalal & Triggs, 2005) — the gradient-cell rendering of the input feature representation. `skimage.feature.hog` returns it via the `visualize=True` flag. This shows what the SVM literally sees — no surrogate model.
2. **Occlusion sensitivity** (Zeiler & Fergus, 2014) — slide a 16×16 mask across the 128×128 input with stride 8, recording the drop in pneumonia probability when each region is masked. Bigger drops mean the SVM leans on that region more.

I run both on two cases: one correct PNEUMONIA call (`person100_bacteria_475.jpeg`) and one misclassified NORMAL (`IM-0001-0001.jpeg`).

**Caveat (saturated baseline):** when the SVM is at probability ≈ 1.0 on a correct case, occluding any single patch can only drop the probability by a tiny amount. The heatmap range collapses (≈ 0.000-0.010 in this run). The misclassified case has more dynamic range and is more informative for the shortcut-learning question.

*Decision provenance · Tier A: HOG visualisation (Dalal & Triggs, 2005); occlusion sensitivity (Zeiler & Fergus, 2014). Tier B: choice of unified XAI tool (occlusion both sides) — comparative-narrative coherence over richer attribution. LIME (Ribeiro et al., 2016) and Grad-CAM (Selvaraju et al., 2017) kept in references as next-step anchors.*


In [ ]:
# HOG visualisation + occlusion sensitivity on the SVM. No helper functions.

# Pick one correct PNEUMONIA case and one misclassified case
idx_correct = int(np.where((svm_pred == y_test) & (y_test == 1))[0][0])
idx_wrong   = int(np.where(svm_pred != y_test)[0][0])
print(f"Correct PNEUMONIA at index {idx_correct}, misclassified at index {idx_wrong}")

OCC_PATCH  = 16
OCC_STRIDE = 8

fig, axes = plt.subplots(2, 2, figsize=(11, 10))

for row_i, (idx, title) in enumerate([(idx_correct, "Correct PNEUMONIA"),
                                      (idx_wrong,   "Misclassified")]):
    # ----------------------------------------------------------------
    # Load the image and run the same preprocessing the SVM expects
    # ----------------------------------------------------------------
    img_path = test_df["path"].iloc[idx]
    img_gray = np.array(Image.open(img_path).convert("L"))
    from skimage import transform as sktransform
    img_gray = sktransform.resize(img_gray, (HOG_SIZE, HOG_SIZE), anti_aliasing=True)
    img_gray = exposure.equalize_adapthist(img_gray, clip_limit=0.03)

    # ----------------------------------------------------------------
    # (a) HOG visualisation — Dalal & Triggs (2005), skimage built-in
    # ----------------------------------------------------------------
    feat, hog_img = hog(
        img_gray,
        orientations=9,
        pixels_per_cell=(8, 8),
        cells_per_block=(2, 2),
        block_norm="L2-Hys",
        feature_vector=True,
        visualize=True,
    )

    ax_hog = axes[row_i, 0]
    ax_hog.imshow(hog_img, cmap="gray")
    border = "#2D6A4F" if "Correct" in title else "#C9302C"
    ax_hog.set_title(f"SVM — HOG visualisation — {title}", color=border, weight="bold")
    ax_hog.axis("off")

    # ----------------------------------------------------------------
    # (b) Occlusion sensitivity — Zeiler & Fergus (2014)
    # ----------------------------------------------------------------
    baseline_feat = feat.reshape(1, -1).astype(np.float32)
    baseline_proba = svm.predict_proba(scaler.transform(baseline_feat))[0, 1]
    print(f"  {title}: baseline P(PNEUMONIA) = {baseline_proba:.4f}")

    H = W = HOG_SIZE
    heatmap = np.zeros((H, W), dtype=np.float32)
    n_patches = 0
    t0 = time.time()
    # Verbose for-loop (no list comprehension):
    for y_top in range(0, H - OCC_PATCH + 1, OCC_STRIDE):
        for x_left in range(0, W - OCC_PATCH + 1, OCC_STRIDE):
            masked = img_gray.copy()
            masked[y_top:y_top + OCC_PATCH,
                   x_left:x_left + OCC_PATCH] = 0.0
            masked_feat = hog(
                masked,
                orientations=9,
                pixels_per_cell=(8, 8),
                cells_per_block=(2, 2),
                block_norm="L2-Hys",
                feature_vector=True,
            ).reshape(1, -1).astype(np.float32)
            masked_proba = svm.predict_proba(scaler.transform(masked_feat))[0, 1]
            drop = baseline_proba - masked_proba
            heatmap[y_top:y_top + OCC_PATCH,
                    x_left:x_left + OCC_PATCH] += drop
            n_patches += 1
    print(f"  {title}: {n_patches} patches, "
          f"heatmap range [{heatmap.min():.4f}, {heatmap.max():.4f}], "
          f"{time.time() - t0:.1f}s")

    ax_occ = axes[row_i, 1]
    ax_occ.imshow(img_gray, cmap="gray")
    im = ax_occ.imshow(heatmap, cmap="seismic",
                       vmin=-abs(heatmap).max(), vmax=abs(heatmap).max(),
                       alpha=0.55)
    ax_occ.set_title(f"SVM — Occlusion sensitivity — {title}", color=border, weight="bold")
    ax_occ.axis("off")
    cbar = plt.colorbar(im, ax=ax_occ, fraction=0.046, pad=0.04)
    cbar.set_label("P(PNEUMONIA) drop when masked", fontsize=9)

plt.tight_layout()
plt.show()


## 13. Explainability — Occlusion sensitivity on the CNN

For the CNN I use the same occlusion sensitivity technique scaled up to the 224x224 input, so both models share one XAI tool. Same Zeiler & Fergus (2014) method as §12, scaled up to the CNN's 224×224 input with a 32×32 mask and stride 16.

The unified-XAI choice is deliberate: the §17 ethics use ("the heatmap lets us check shortcut learning on both models") is cleaner if both models are explained with the same tool. Grad-CAM is richer (gradient-based rather than a black-box probe) and stays in the references as a next-step anchor — §21 limitations explicitly lists "add LIME on SVM and Grad-CAM on CNN for richer attribution" as future work.

*Decision provenance · Tier A: occlusion sensitivity (Zeiler & Fergus, 2014). Tier B: unified-XAI choice (same tool both sides) — comparative-narrative coherence over per-model richness. Grad-CAM (Selvaraju et al., 2017) + CheXNet (Rajpurkar et al., 2017) kept as next-step anchors.*


In [ ]:
# occlusion sensitivity on the CNN. No helper functions.

idx_correct = int(np.where((cnn_pred == y_test_cnn) & (y_test_cnn == 1))[0][0])
idx_wrong   = int(np.where(cnn_pred != y_test_cnn)[0][0])
print(f"Correct PNEUMONIA at index {idx_correct}, misclassified at index {idx_wrong}")

OCC_CNN_PATCH  = 32
OCC_CNN_STRIDE = 16

fig, axes = plt.subplots(2, 1, figsize=(8, 12))

for row_i, (idx, title) in enumerate([(idx_correct, "Correct PNEUMONIA"),
                                      (idx_wrong,   "Misclassified")]):
    img_pil = Image.open(test_df["path"].iloc[idx]).convert("RGB").resize((IMG_SIZE, IMG_SIZE))
    img_uint8 = np.array(img_pil)

    baseline_batch = preprocess_input(img_uint8.astype(np.float32)[None, ...])
    baseline_proba = float(cnn.predict(baseline_batch, verbose=0)[0, 0])
    print(f"  {title}: baseline P(PNEUMONIA) = {baseline_proba:.4f}")

    H = W = IMG_SIZE
    heatmap = np.zeros((H, W), dtype=np.float32)
    n_patches = 0
    t0 = time.time()
    # Verbose for-loop (no batching, no list comprehension):
    for y_top in range(0, H - OCC_CNN_PATCH + 1, OCC_CNN_STRIDE):
        for x_left in range(0, W - OCC_CNN_PATCH + 1, OCC_CNN_STRIDE):
            masked_uint8 = img_uint8.copy()
            masked_uint8[y_top:y_top + OCC_CNN_PATCH,
                         x_left:x_left + OCC_CNN_PATCH, :] = 128
            masked_batch = preprocess_input(masked_uint8.astype(np.float32)[None, ...])
            masked_proba = float(cnn.predict(masked_batch, verbose=0)[0, 0])
            drop = baseline_proba - masked_proba
            heatmap[y_top:y_top + OCC_CNN_PATCH,
                    x_left:x_left + OCC_CNN_PATCH] += drop
            n_patches += 1
    print(f"  {title}: {n_patches} patches, "
          f"heatmap range [{heatmap.min():.4f}, {heatmap.max():.4f}], "
          f"{time.time() - t0:.1f}s")

    ax = axes[row_i]
    ax.imshow(img_uint8)
    im = ax.imshow(heatmap, cmap="seismic",
                   vmin=-abs(heatmap).max(), vmax=abs(heatmap).max(),
                   alpha=0.55)
    border = "#2D6A4F" if "Correct" in title else "#C9302C"
    ax.set_title(f"CNN — Occlusion sensitivity — {title}", color=border, weight="bold")
    ax.axis("off")
    cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label("P(PNEUMONIA) drop when masked", fontsize=9)

plt.tight_layout()
plt.show()


## 14. Bias and subgroup analysis

Aggregate accuracy can hide real disparities across subpopulations. Without explicit demographic labels in the dataset, I check three intrinsic splits derived from the data itself:

1. **Disease subtype** (within PNEUMONIA): viral vs bacterial, taken from the filename convention.
2. **Brightness terciles**: average pixel intensity, split into low / mid / high.
3. **Image-size terciles**: total pixel count, split into small / mid / large.

A model that performs well on aggregate but poorly on a subgroup is biased in a way the headline metrics can't show.

*Decision provenance · Tier A: cross-site degradation evidence (Zech et al., 2018). Tier C: subgroup splits run on this dataset (subtype, brightness, image size).*


In [ ]:
# subgroup_metrics inlined into the result-collection loop (no helper).

diag = test_df.copy().reset_index(drop=True)
diag["svm_pred"] = svm_pred
diag["cnn_pred"] = cnn_pred[:len(diag)]
diag["luminance"] = [float(np.array(Image.open(p).convert("L")).mean())
                     for p in diag["path"]]
diag["pixels"]    = [Image.open(p).size[0] * Image.open(p).size[1]
                     for p in diag["path"]]
diag["lum_bin"]  = pd.qcut(diag["luminance"], 3, labels=["low", "mid", "high"])
diag["size_bin"] = pd.qcut(diag["pixels"],    3, labels=["small", "mid", "large"])
diag["subtype"]  = diag["path"].apply(
    lambda p: "viral" if "virus" in Path(p).name.lower() else "bacterial")

rows = []

# Subtype within PNEUMONIA
for subtype, df_g in diag[diag.label == 1].groupby("subtype"):
    rows.append({
        "Subgroup":         f"PNEU subtype: {subtype}",
        "n":                len(df_g),
        "SVM macro-F1":     f1_score(df_g["label"], df_g["svm_pred"], average="macro", zero_division=0),
        "CNN macro-F1":     f1_score(df_g["label"], df_g["cnn_pred"], average="macro", zero_division=0),
        "SVM PNEU recall":  recall_score(df_g["label"], df_g["svm_pred"], pos_label=1, zero_division=0),
        "CNN PNEU recall":  recall_score(df_g["label"], df_g["cnn_pred"], pos_label=1, zero_division=0),
    })

# Brightness terciles
for binname, df_g in diag.groupby("lum_bin"):
    rows.append({
        "Subgroup":         f"Brightness: {binname}",
        "n":                len(df_g),
        "SVM macro-F1":     f1_score(df_g["label"], df_g["svm_pred"], average="macro", zero_division=0),
        "CNN macro-F1":     f1_score(df_g["label"], df_g["cnn_pred"], average="macro", zero_division=0),
        "SVM PNEU recall":  recall_score(df_g["label"], df_g["svm_pred"], pos_label=1, zero_division=0),
        "CNN PNEU recall":  recall_score(df_g["label"], df_g["cnn_pred"], pos_label=1, zero_division=0),
    })

# Image size terciles
for binname, df_g in diag.groupby("size_bin"):
    rows.append({
        "Subgroup":         f"Image size: {binname}",
        "n":                len(df_g),
        "SVM macro-F1":     f1_score(df_g["label"], df_g["svm_pred"], average="macro", zero_division=0),
        "CNN macro-F1":     f1_score(df_g["label"], df_g["cnn_pred"], average="macro", zero_division=0),
        "SVM PNEU recall":  recall_score(df_g["label"], df_g["svm_pred"], pos_label=1, zero_division=0),
        "CNN PNEU recall":  recall_score(df_g["label"], df_g["cnn_pred"], pos_label=1, zero_division=0),
    })

subgroup_df = pd.DataFrame(rows)
print(subgroup_df.to_string(index=False, float_format=lambda v: f"{v:.4f}"))


## 15. Comparison test 1 — HOG image size

The HOG image-resize size is a Tier-C decision (no clear external prescription). I tested 64, 128, and 192 pixels with everything else held constant — same SVM hyperparameters, same train split. v11.6 methodology fix: the comparison evaluates on the **validation** set (not test) to avoid test-set leakage; the test set is used only once, on the chosen size, as the single-shot final report on §9.

| HOG_SIZE | Features | Val Macro-F1 | Val Accuracy | Val ROC-AUC | Train time |
|----------|---------:|-------------:|-------------:|------------:|-----------:|
| 64       |    1,764 |       0.9579 |       0.9675 |      0.9940 |       185s |
| **128**  |    8,100 |   **0.9638** |       0.9723 |      0.9959 |       409s |
| 192      |   19,044 |       0.9605 |       0.9694 |      0.9952 |      1028s |

128 wins on the validation set by a small but consistent margin (~0.6 macro-F1 points over 64; also wins on accuracy and ROC-AUC). The headline SVM pipeline uses 128, anchored in this clean comparison.

**Caveat:** the comparison holds SVM hyperparameters fixed at the headline-pipeline winners C=1, gamma=0.0001 across all three sizes (controlled-variable design — only HOG_SIZE varies). Different feature dimensions probably want different C and gamma, so a fully principled comparison would re-tune per size. Documented as a limitation and listed in §21 next steps.

v11.8 used a small `extract_hog_at(path, size)` helper. v11.9 inlines the HOG extraction body into the size loop below.

*Decision provenance · Tier C: HOG image size 128×128 — selected on val_df only at fixed C=1, gamma=0.0001 (headline-pipeline winners); test result reported once, on §9.*


In [ ]:
# inline HOG extraction (no extract_hog_at helper).

from skimage import transform as sktransform2

hog_size_results = []
for size in [64, 128, 192]:
    print(f"Testing HOG_SIZE = {size}...")
    t0 = time.time()

    # Train HOG features at this size — inline
    Xtr_list = []
    for p in train_df["path"]:
        img = Image.open(p).convert("L").resize((size, size))
        img = np.array(img) / 255.0
        img = exposure.equalize_adapthist(img, clip_limit=0.03)
        Xtr_list.append(hog(img, orientations=9, pixels_per_cell=(8, 8),
                            cells_per_block=(2, 2), block_norm="L2-Hys"))
    Xtr = np.array(Xtr_list, dtype=np.float32)
    del Xtr_list

    # Val HOG features at this size — inline (v11.6 fix: evaluate on val, not test)
    Xva_list = []
    for p in val_df["path"]:
        img = Image.open(p).convert("L").resize((size, size))
        img = np.array(img) / 255.0
        img = exposure.equalize_adapthist(img, clip_limit=0.03)
        Xva_list.append(hog(img, orientations=9, pixels_per_cell=(8, 8),
                            cells_per_block=(2, 2), block_norm="L2-Hys"))
    Xva = np.array(Xva_list, dtype=np.float32)
    del Xva_list

    sc = StandardScaler().fit(Xtr)
    Xtr, Xva = sc.transform(Xtr), sc.transform(Xva)
    m = SVC(kernel="rbf", class_weight="balanced", probability=True,
            C=svm.C, gamma=svm.gamma, random_state=SEED)
    m.fit(Xtr, y_train)
    pr = (m.predict_proba(Xva)[:, 1] >= 0.5).astype(int)
    hog_size_results.append({
        "HOG_SIZE":      size,
        "Features":      Xtr.shape[1],
        "Time (s)":      int(time.time() - t0),
        "Val Accuracy":  accuracy_score(y_val, pr),
        "Val Macro-F1":  f1_score(y_val, pr, average="macro"),
        "Val ROC-AUC":   roc_auc_score(y_val, m.predict_proba(Xva)[:, 1]),
    })

hog_size_df = pd.DataFrame(hog_size_results)
print(hog_size_df.to_string(index=False, float_format=lambda v: f"{v:.4f}"))


## §15.5 — Model demonstration case

This is the same case shown on slide 18 of the presentation. The image is a held-out paediatric NORMAL chest X-ray (`NORMAL2-IM-0135-0001.jpeg`) from the Kermany 2018 test set, never seen during training or hyperparameter tuning. Both models load from the persisted weights (`archive/_run_artifacts_run1/models/`) — the same trained instance that produced every other v12 metric in this notebook.

The disagreement pattern on this single image (SVM confidently PNEUMONIA, CNN correctly NORMAL) is the empirical case for the SVM-as-shadow-model design in §17 recommendation: when the cheap classical SVM disagrees with the deep CNN, the disagreement itself is the signal a radiologist should investigate.

Threshold = 0.5 (default).


In [ ]:
# Load the demo image and run both models end-to-end
from pathlib import Path
import numpy as np
from PIL import Image
from skimage import transform, exposure
from skimage.feature import hog
from tensorflow.keras.applications.resnet50 import preprocess_input

DEMO_IMAGE = Path("chest_xray/test/NORMAL/NORMAL2-IM-0135-0001.jpeg")
print(f"Demo image: {DEMO_IMAGE.name}")
print(f"True label: NORMAL")
print()

# SVM HOG+CLAHE+scaler+predict_proba pipeline (matches §6 / §9 headline)
img_gray = np.array(Image.open(DEMO_IMAGE).convert("L"))
img_gray = transform.resize(img_gray, (128, 128), anti_aliasing=True)
img_gray = exposure.equalize_adapthist(img_gray, clip_limit=0.03)
hog_feat = hog(img_gray, orientations=9, pixels_per_cell=(8, 8),
               cells_per_block=(2, 2), block_norm="L2-Hys", feature_vector=True)
svm_prob = svm.predict_proba(scaler.transform(hog_feat.reshape(1, -1)))[0, 1]

# CNN ResNet50 frozen-base pipeline (matches §10 headline)
img_rgb = np.array(Image.open(DEMO_IMAGE).convert("RGB").resize((224, 224)))
batch = preprocess_input(img_rgb.astype(np.float32)[None, ...])
cnn_prob = float(cnn.predict(batch, verbose=0)[0, 0])

svm_label = "PNEUMONIA" if svm_prob >= 0.5 else "NORMAL"
cnn_label = "PNEUMONIA" if cnn_prob >= 0.5 else "NORMAL"

print(f"SVM:  P(PNEUMONIA) = {svm_prob:.4f}  ->  {svm_label}")
print(f"CNN:  P(PNEUMONIA) = {cnn_prob:.4f}  ->  {cnn_label}")
print(f"True label: NORMAL")
print()
print(f"Disagreement: SVM says {svm_label}, CNN says {cnn_label}.")
print(f"Slide 18 reports SVM 0.9549 PNEUMONIA, CNN 0.3789 NORMAL — same case, same models.")


In [ ]:
# Visualise the demo case
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 1, figsize=(9, 9))
ax.imshow(Image.open(DEMO_IMAGE), cmap="gray")
ax.axis("off")
ax.set_title(
    f"Held-out test image: NORMAL2-IM-0135-0001.jpeg

"
    f"SVM:  P(PNEUMONIA) = {svm_prob:.4f}  ->  {svm_label}
"
    f"CNN:  P(PNEUMONIA) = {cnn_prob:.4f}  ->  {cnn_label}

"
    f"True label: NORMAL",
    fontsize=12, loc="left", pad=14, family="monospace"
)
plt.tight_layout()
plt.show()


## 16. Comparison test 2 — CNN augmentation on/off

Does data augmentation help on this dataset? I train the same CNN architecture twice with the same training schedule (frozen base, head trained for 5 epochs), only toggling the augmentation pipeline on or off. **Important methodological note**: this is a single-seed comparison, and a small effect size can easily be inside run-to-run variance for ResNet50 head training at 5 epochs (Bouthillier et al., 2019). Treat the result as illustrative, not definitive — multi-seed averaging would be needed to settle the question.

*Decision provenance · Tier C: CNN augmentation comparison on this dataset — two runs, opposite orderings, both within single-seed variance (Bouthillier et al., 2019). Tier C correctly identifies when project evidence is underpowered: the framework also tells me when evidence is not strong enough to settle the question.*


In [ ]:
aug_results = []
for aug_label in ["with", "without"]:
    print(f"Augmentation: {aug_label}")
    aug_train = make_gen(train_df, augment=(aug_label == "with"), shuffle=True)
    aug_val   = make_gen(val_df,   augment=False, shuffle=False)
    aug_test  = make_gen(test_df,  augment=False, shuffle=False)

    m, _ = build_cnn()
    m.compile(optimizer=optimizers.Adam(1e-3),
              loss="binary_crossentropy",
              metrics=["accuracy", tf.keras.metrics.AUC(name="auc")])
    m.fit(aug_train, validation_data=aug_val, epochs=5,
          class_weight=class_weights, verbose=1)

    proba = m.predict(aug_test, verbose=0).ravel()
    pred  = (proba >= 0.5).astype(int)
    y_eval = test_df["label"].values[:len(pred)]
    aug_results.append({
        "Augmentation": aug_label,
        "Accuracy":  accuracy_score(y_eval, pred),
        "Macro-F1":  f1_score(y_eval, pred, average="macro"),
        "ROC-AUC":   roc_auc_score(y_eval, proba),
    })
    tf.keras.backend.clear_session()

aug_df = pd.DataFrame(aug_results)
print(aug_df.to_string(index=False, float_format=lambda v: f"{v:.4f}"))


## 17. Ethical considerations

The ethics analysis here is grounded in the specific dataset rather than abstract principles.

**Data privacy and PII.** The data is paediatric, from a single hospital, retrospectively collected. DICOM Supplement 142 (the standard de-identification profile in clinical imaging) strips header identifiers, but text burned into the image pixels themselves bypasses header stripping — any deployed system would also need OCR-based pixel-text redaction. Public Kaggle redistribution of medical imaging does not re-state patient consent, so any deployment from this dataset would require fresh data-governance review.

**Fairness and error costs.** A missed pneumonia (false negative) is potentially fatal in a paediatric patient. A false positive costs clinician minutes and may contribute to antibiotic stewardship pressure at scale. The default 0.5 threshold treats these errors as equally costly, which Provost and Fawcett (2001) point out is rarely operationally correct. From the actual threshold sweep on this CNN: at threshold 0.5 we have 114 false positives and 2 missed pneumonias on the test set. Raising the threshold to 0.80 reduces false positives to 84 (a 30-case reduction, 26%) at the cost of 1 additional missed pneumonia (2 → 3), while keeping PNEUMONIA recall at 0.9923 — above the 0.99 operating target. This happens because the CNN's predictions are saturated towards the extremes (few scores in the middle), so a threshold increase doesn't crater recall the way classical-ML intuition might predict. The ethics conclusion: the default 0.5 over-predicts pneumonia; a clinician-selected operating point at 0.80 reduces the false-positive workload substantially while keeping recall in the critical 0.99+ range. The decision should sit with clinicians, not with the model defaults.

**Bias and harm.** Three risks specific to this dataset: (1) population transfer — paediatric to adult silently degrades performance; (2) equipment shift — different X-ray machines change the input distribution; (3) shortcut learning — a model can latch on to chest tubes or text annotations rather than pathology (Zech et al., 2018), which is what the occlusion-sensitivity heatmap helps check visually — and in this run the heatmap actually surfaces shoulder/edge attention rather than lung-region attention on both correct and misclassified cases, providing direct empirical evidence of the Zech 2018 risk on this trained instance. The brightness sensitivity I found in the subgroup analysis is a real bias signal worth flagging in deployment monitoring. Each of these is a deployment veto if not actively monitored.

*Decision provenance · Tier A: cost-sensitive threshold framing (Provost & Fawcett, 2001); shortcut learning (Zech et al., 2018). Tier B: de-identification standard (DICOM Supplement 142, NEMA 2011). Tier C: threshold sweep numbers measured on this dataset (§11).*


## 18. Deployment plan

**Deployment architecture.** The presentation slide 23 shows a UML deployment / component diagram with three named planes (Hospital data, Cloud inference, MLOps operations) plus a Governance annotation. The text below summarises the same architecture in prose for the marker's audit trail. Stereotypes follow OMG UML 2.5 conventions; the agent / monitor / model-registry pattern anchors in Sculley et al. (2015) on hidden technical debt in machine learning systems. The model registry component shown on the diagram corresponds directly to the persisted trained artefacts in `archive/_run_artifacts_run1/models/` — that is Decision 21 operationalised.

This is a discussion of how the recommended model could be deployed in a real clinical setting, not an implementation. The target context I would design for is a paediatric outpatient triage assistant.

**Architecture.** ResNet50 served behind an AWS SageMaker endpoint. Input images flow from the hospital's PACS through a DICOM-to-JPEG adapter into the endpoint. The model output (predicted class plus probability) is returned to a clinician worklist UI alongside an occlusion-sensitivity heatmap (the same XAI tool used in evaluation), available on demand for clinician review of borderline cases. Routine UI uses lighter-weight Grad-CAM (planned in next-step pipeline) — occlusion is ~100× more expensive per explanation than Grad-CAM at inference time, so for a deployment serving live triage volume the cost difference matters. Modest GPU-backed cloud hosting handles the inference load; cost scales with the volume of clinics served.

**Monitoring.** Three monitors run continuously: input drift via a Kolmogorov–Smirnov test on luminance and image size distributions; weekly performance evaluation on a relabelled sample of recent inferences; and weekly per-subgroup performance metrics (the same brightness, size, and disease-subtype splits used in §14).

**Retraining triggers.** Three explicit triggers: a KS-test breach for seven consecutive days, a macro-F1 drop greater than 3 percentage points week-on-week, or a new equipment vendor introduced into the data stream.

**Regulatory path.** This would likely fall under Class IIa per MDR Annex VIII Rule 11, applying the MDCG 2019-11 software-qualification-and-classification guidance — but that is a one-line architectural guess, not a determination. A deployment workstream would need a formal qualification-and-classification analysis to confirm class. Out of scope for this assignment but the next concrete blocker for any real deployment.


## 19. Recommendation

**Deploy ResNet50, with safeguards.**

The CNN substantially outperforms the SVM on the metrics that matter under class imbalance: macro-F1 of 0.7721 against 0.6846 (a 9-point gap), NORMAL recall of 0.51 against 0.37 (CNN +14 points; SVM still over-predicting pneumonia rather than separating classes), and a much smaller gap between cross-validation and test scores. Occlusion-sensitivity heatmaps surface a shortcut-learning concern on both models (shoulder/edge attention rather than lung-region) — this is empirical evidence supporting the §21 next-step of adding richer XAI (LIME on SVM, Grad-CAM on CNN) plus calibration analysis.

I would not entirely retire the SVM. It is cheap to run alongside the CNN as a shadow model, and disagreements between the two models are useful priority-flagging signals for radiologist review. Its feature pipeline is also fully inspectable, which helps in any regulatory conversation.

**Five non-negotiable safeguards before any of this goes near a patient:**

1. **Clinician-selected threshold** with a documented operating target — for example, ≥0.99 PNEUMONIA recall if the false-positive workload is acceptable (per the threshold sweep in §11). The threshold is not a hard-coded constant.
2. **Region-importance heatmap (occlusion sensitivity) AVAILABLE ON DEMAND for clinician review**; lighter Grad-CAM is the preferred routine UI surface (planned next-step).
3. **Weekly subgroup monitoring** across brightness, disease subtype, and equipment vendor.
4. **Explicit rejection of adult X-rays** as out-of-distribution — the model was trained on paediatric data only.
5. **Human-in-the-loop, always** — the model is decision support, not an autonomous diagnostic.

Without all five, this stays a prototype, not a deployed device.


## 20. Critical reflections

The decision-provenance framework made the project auditable, but it also exposed where the evidence was weaker. The reflections below are what I would do differently if starting again — and several of them are places where the framework itself surfaced a limitation rather than glossing over it.

**Frozen-base via doc reading.** My initial design read as two-phase fine-tuning, but Keras handles `trainable` flags asymmetrically on nested models: when the parent base has `trainable = False`, setting `layer.trainable = True` on individual children does not engage gradients on those children. The unfreeze loop in earlier code was therefore inert, and what actually ran was a frozen-base feature extractor with the head trained at two learning rates. On closer reading of the TensorFlow documentation I surfaced this and reframed the model honestly here as a frozen-base baseline (still standard in medical imaging — Cheplygina et al., 2019) rather than re-running under deadline pressure.

**Simpler XAI by design (v11.9 reflection).** I replaced LIME on the SVM with HOG visualisation + occlusion sensitivity, and Grad-CAM on the CNN with occlusion sensitivity, so both models use one tool (Zeiler & Fergus, 2014). The trade-off: LIME and Grad-CAM are richer attribution methods than a black-box occlusion probe, but the unified-tool comparative narrative reads cleaner across §12 / §13 / §17 / §18 / §19. Code transparency wins too — no surrogate model, no gradient-tape plumbing through nested ResNet50. LIME and Grad-CAM stay in the references as next-step anchors (§21).

**Saving the model is methodology, not housekeeping.** During development we hit a traceability incident — earlier model weights weren't preserved as a versioned `.keras` artefact, so re-running any analysis required a fresh re-train, The fix was a single end-to-end re-run; the prevention going forward is disciplined model-weight persistence. v12 saves `cnn_resnet50_<timestamp>.keras` unconditionally.

**Learning the domain before designing the pipeline.** I came in without prior experience in image processing or chest-X-ray imaging. To make the design choices defensible — image resize, CLAHE, HOG parameters, augmentation, subgroup splits, the entire ethics framing around shortcut learning — I had to invest learning time in two domain layers first: image-processing fundamentals (pixels, resolution, what HOG features capture), and chest-X-ray specifics (bacterial vs viral presentations, text burn-in and chest-tube shortcut risk per Zech 2018, paediatric-only single-hospital limits). Without that learning the code would still run but every choice would be arbitrary.

**HOG-size leakage caught.** Peer review of v11.5 caught that my HOG-size comparison test evaluated on the test set, then the headline pipeline used the winner — that is methodological leakage. v11.6 fixes this by re-running the comparison cleanly on the validation set only (§15), then evaluating the chosen size once on the test set. Outcome: HOG_SIZE=128 wins on validation by a small but consistent margin, and the new SVM headline numbers (0.6846 macro-F1, 0.37 NORMAL recall) reflect that clean methodology.

**Calibration.** The CNN's sigmoid output is not a true probability — both example XAI cases on slide 17 saturate at P=1.0000, including the confidently-wrong NORMAL case. Calibration analysis is the next concrete step: reliability curves, Expected Calibration Error (ECE), Brier score, and temperature scaling on the CNN probabilities; calibration check on the SVM Platt-scaled outputs (Platt, 1999); plus statistical significance testing on the head-to-head comparison (bootstrap CI on macro-F1; McNemar's test on paired predictions).

**Single dataset.** This entire study is on one hospital's data. Zech et al. (2018) showed a chest-X-ray DL model dropping from 0.93 AUC on internal data to 0.74 AUC on a different hospital's data — a kind of cross-site degradation that is invisible from a single-site study like this one. External validation on RSNA Pneumonia or NIH ChestX-ray14 would be the obvious next step.

**Generalisation gap finding.** The SVM's CV (0.961) → test (0.685) is a 27-point drop. The CNN's CV (0.8933) → test (0.7721) is a 12-point drop. Neither is the 'near-zero gap' I claimed in earlier writeups — that was a v8-era finding from a different trained instance. But the CNN's gap is roughly 2× cleaner than the SVM's, and that comparative point still grounds the recommendation.

**Augmentation comparison.** I ran comparison test 2 twice on different occasions and got opposite orderings — augmentation helped in one run, hurt in the other. Both within typical run-to-run variance per Bouthillier et al. (2019). Multi-seed averaging is what would be needed to settle the question; a single-seed test does not have the statistical power to do so. I report this honestly because cherry-picking would be the wrong move.

**Closing reflection.** What felt natural from my enterprise-architecture background was the systems side: traceability, governance, deployment risk, monitoring, and model-artefact control. The three-tier decision-provenance framework on §1.5 is that architecture instinct applied to ML uncertainty — a way to make every modelling choice auditable rather than treating them as opaque expert judgement. What I genuinely had to learn was ML-specific: validation leakage, stochastic CNN training, frozen-base transfer learning, the limits of XAI when probabilities saturate, calibration, and disciplined model-weight persistence. That is why the recommendation is cautious — ResNet50 here is the stronger triage candidate, not a clinical product. External validation, monitoring, threshold governance, and clinician oversight are non-negotiable before any patient-facing use.


## 21. Limitations and next steps

Eleven concrete next steps if the project continued:

1. **External validation** on a different cohort — RSNA Pneumonia (Radiological Society of North America, 2018) or NIH ChestX-ray14 (Wang et al., 2017) — different equipment, different population.
2. **DenseNet121 head-to-head** against ResNet50 — re-architect to match CheXNet precedent.
3. **Re-run a true two-phase fine-tune** with the Keras nested-model fix — clone the base, set the inner-layer trainable flags before re-attaching, recompile, and re-run — so we get an actual fine-tuning result rather than the frozen-base baseline reported in §7.
4. **Per-size HOG hyperparameter retuning** — the v11.6 size comparison holds C and gamma fixed across 64 / 128 / 192. Different feature dimensions probably want different hyperparameters; a fully principled comparison would re-tune per size before claiming a winner.
5. **Multi-seed averaging** on the comparison tests — five to ten seeds per arm to settle whether augmentation helps.
6. **Calibration analysis** (Platt scaling) and **statistical significance testing** (bootstrap confidence intervals on macro-F1 or McNemar's test on paired predictions).
7. **Prospective rather than retrospective evaluation** — every number in this notebook is retrospective; a clinical claim requires prospective data.
8. **Add LIME on the SVM and Grad-CAM on the CNN** for richer attribution than the unified occlusion probe used here — anchored in Ribeiro et al. (2016) and Selvaraju et al. (2017) / Rajpurkar et al. (2017) / CheXNet. We chose a simpler unified-XAI design for comparative-narrative cleanliness; richer attribution is a clean next step.
9. **Versioned model-weight artefacts** (model registry pattern, training-config snapshot, content hash) — operational discipline per the v12 traceability lesson. The v11.9 full-trace re-run was forced by an earlier failure to persist v8 weights; this is the prevention rule going forward.
10. **Ablate the Phase-2 LR schedule itself.** The v12 augmentation comparison (5-epoch single-LR augmented training) reached test macro-F1 0.892, while the headline two-stage training (10 epochs at 1e-3 + 4 epochs at 1e-4) reached only 0.772 on the same architecture and data. The Phase-2 schedule may not be helping — possibly hurting — and is worth ablating in a future iteration.



## References (Harvard Cite Them Right)

Bouthillier, X., Laurent, C. and Vincent, P. (2019) 'Unreproducible Research is Reproducible', in *Proceedings of the 36th International Conference on Machine Learning (ICML)*.

Cheplygina, V., de Bruijne, M. and Pluim, J.P.W. (2019) 'Not-so-supervised: A survey of semi-supervised, multi-instance, and transfer learning in medical image analysis', *Medical Image Analysis*, 54, pp. 280–296.

Dalal, N. and Triggs, B. (2005) 'Histograms of oriented gradients for human detection', in *2005 IEEE Computer Society Conference on Computer Vision and Pattern Recognition (CVPR'05)*, vol. 1, pp. 886–893.

DICOM Standards Committee (2011) *Supplement 142: Clinical Trial De-identification Profiles*. Rosslyn, VA: NEMA.

European Parliament (2017) *Regulation (EU) 2017/745 on Medical Devices (MDR)*. Official Journal of the European Union, L117, 5 May 2017.

Amazon Web Services (2024) *Amazon SageMaker Developer Guide*. Available at: https://docs.aws.amazon.com/sagemaker/ (Accessed: April 2026).

He, K., Zhang, X., Ren, S. and Sun, J. (2016) 'Deep residual learning for image recognition', in *Proceedings of the IEEE Conference on Computer Vision and Pattern Recognition*, pp. 770–778.

Hevner, A.R., March, S.T., Park, J. and Ram, S. (2004) 'Design science in information systems research', *MIS Quarterly*, 28(1), pp. 75–105.

Hsu, C-W., Chang, C-C. and Lin, C-J. (2003) *A Practical Guide to Support Vector Classification*. Department of Computer Science, National Taiwan University, Technical Report.

Kermany, D.S., Goldbaum, M., Cai, W. *et al.* (2018) 'Identifying medical diagnoses and treatable diseases by image-based deep learning', *Cell*, 172(5), pp. 1122–1131.

King, G. and Zeng, L. (2001) 'Logistic regression in rare events data', *Political Analysis*, 9(2), pp. 137–163.

Kingma, D.P. and Ba, J. (2014) 'Adam: A method for stochastic optimization', in *International Conference on Learning Representations*.

Kohavi, R. (1995) 'A study of cross-validation and bootstrap for accuracy estimation and model selection', in *Proceedings of the 14th International Joint Conference on Artificial Intelligence*, vol. 2, pp. 1137–1143.

Lin, M., Chen, Q. and Yan, S. (2014) 'Network in network', in *International Conference on Learning Representations (ICLR)*.

Medical Device Coordination Group (2019) *MDCG 2019-11: Guidance on Qualification and Classification of Software in Regulation (EU) 2017/745 — MDR — and Regulation (EU) 2017/746 — IVDR*. European Commission, October 2019.

Pizer, S.M., Amburn, E.P., Austin, J.D. *et al.* (1987) 'Adaptive histogram equalization and its variations', *Computer Vision, Graphics, and Image Processing*, 39(3), pp. 355–368.

Platt, J. (1999) 'Probabilistic outputs for support vector machines and comparisons to regularized likelihood methods', in Smola, A.J. *et al.* (eds) *Advances in Large Margin Classifiers*. Cambridge, MA: MIT Press, pp. 61–74.

Prechelt, L. (1998) 'Early stopping — but when?', in Orr, G.B. and Müller, K-R. (eds) *Neural Networks: Tricks of the Trade*. Berlin: Springer, pp. 55–69.

Provost, F. and Fawcett, T. (2001) 'Robust classification for imprecise environments', *Machine Learning*, 42(3), pp. 203–231.

Radiological Society of North America (2018) *RSNA Pneumonia Detection Challenge dataset*. Available at: https://www.rsna.org/rsnai/ai-image-challenge/rsna-pneumonia-detection-challenge-2018 (Accessed: April 2026).

Rajpurkar, P., Irvin, J., Zhu, K. *et al.* (2017) 'CheXNet: radiologist-level pneumonia detection on chest X-rays with deep learning', *arXiv:1711.05225*.

Ribeiro, M.T., Singh, S. and Guestrin, C. (2016) "'Why should I trust you?': Explaining the predictions of any classifier", in *Proceedings of the 22nd ACM SIGKDD International Conference on Knowledge Discovery and Data Mining*, pp. 1135–1144. *(LIME — kept as a next-step anchor in v11.9; replaced for the SVM by HOG visualisation + occlusion sensitivity to keep the comparison-pair simpler.)*

Sackett, D.L., Rosenberg, W.M.C., Gray, J.A.M., Haynes, R.B. and Richardson, W.S. (1996) 'Evidence based medicine: what it is and what it isn't', *BMJ*, 312(7023), pp. 71–72.

Selvaraju, R.R., Cogswell, M., Das, A. *et al.* (2017) 'Grad-CAM: Visual explanations from deep networks via gradient-based localization', in *Proceedings of the IEEE International Conference on Computer Vision (ICCV)*, pp. 618–626. *(Grad-CAM — kept as a next-step anchor in v11.9; replaced for the CNN by occlusion sensitivity so both models share the same XAI tool.)*

Shorten, C. and Khoshgoftaar, T.M. (2019) 'A survey on image data augmentation for deep learning', *Journal of Big Data*, 6(60), pp. 1–48.

Srivastava, N., Hinton, G., Krizhevsky, A., Sutskever, I. and Salakhutdinov, R. (2014) 'Dropout: A simple way to prevent neural networks from overfitting', *Journal of Machine Learning Research*, 15(1), pp. 1929–1958.

TensorFlow / Keras (2024) *Transfer learning & fine-tuning* and *tf.keras.Model.trainable*. Official documentation. Available at: https://www.tensorflow.org/guide/keras/transfer_learning (Accessed: April 2026).

UNICEF (2024) *Pneumonia in children — global child mortality data*. Available at: https://data.unicef.org/topic/child-health/pneumonia/ (Accessed: April 2026).

Wang, X., Peng, Y., Lu, L. *et al.* (2017) 'ChestX-ray8: Hospital-scale chest X-ray database and benchmarks on weakly-supervised classification and localization of common thorax diseases', in *Proceedings of the IEEE Conference on Computer Vision and Pattern Recognition (CVPR)*, pp. 2097–2106.

Yosinski, J., Clune, J., Bengio, Y. and Lipson, H. (2014) 'How transferable are features in deep neural networks?', in *Advances in Neural Information Processing Systems 27*, pp. 3320–3328.

Zech, J.R., Badgeley, M.A., Liu, M. *et al.* (2018) 'Variable generalization performance of a deep learning model to detect pneumonia in chest radiographs: A cross-sectional study', *PLOS Medicine*, 15(11), e1002683.

Zeiler, M.D. and Fergus, R. (2014) 'Visualizing and understanding convolutional networks', in *Computer Vision — ECCV 2014. Lecture Notes in Computer Science, vol. 8689*. Cham: Springer, pp. 818–833. doi:10.1007/978-3-319-10590-1_53. *(Occlusion sensitivity — used in v11.9 for both SVM and CNN as the unified XAI method.)*

Zuiderveld, K. (1994) 'Contrast Limited Adaptive Histogram Equalization', in Heckbert, P.S. (ed.) *Graphics Gems IV*. San Diego, CA: Academic Press, pp. 474–485.
